In [1]:
import pandas as pd
import numpy as np
import os
from sklearn.utils import shuffle
import seaborn as sns
import matplotlib.pyplot as plt

# 1. SETUP PATHS
# We check both the 'data/chunks' folder and the current folder
POSSIBLE_PATHS = ['data/chunks', '.'] 
FILENAMES = ['enriched_part_1_early.csv', 'enriched_part_2_mid.csv', 'enriched_part_3_late.csv']

df_list = []

print(f"📂 Current Working Directory: {os.getcwd()}")
print("🔎 Looking for enriched files...")

for fname in FILENAMES:
    file_found = False
    for path in POSSIBLE_PATHS:
        full_path = os.path.join(path, fname)
        if os.path.exists(full_path):
            try:
                # Load and keep only relevant columns
                t = pd.read_csv(full_path, on_bad_lines='skip')
                
                # Check if columns exist before filtering
                required_cols = ['latitude', 'longitude', 'acq_date', 'temp', 'humidity', 'wind']
                if all(col in t.columns for col in required_cols):
                    t = t[required_cols]
                    df_list.append(t)
                    print(f"   ✅ Loaded: {full_path} ({len(t)} rows)")
                    file_found = True
                    break # Stop looking for this file
                else:
                    print(f"   ⚠️ {fname} exists but is missing columns. Skipping.")
            except Exception as e:
                print(f"   ❌ Error reading {fname}: {e}")
    
    if not file_found:
        print(f"   ❌ {fname} NOT FOUND in data/chunks or current folder.")

# CRITICAL STOP
if not df_list: 
    print("\n❌ STOPPING: No data loaded.")
    print("👉 Please make sure your 'enriched_part_X.csv' files are inside the 'data/chunks' folder.")
    exit()

# 2. MERGE
fire_df = pd.concat(df_list, ignore_index=True)
fire_df.dropna(subset=['temp', 'humidity', 'wind'], inplace=True)
fire_df['fire_detected'] = 1
print(f"\n🔥 Total verified fires: {len(fire_df)}")

# 3. PHYSICS-SAFE "SAFE POINTS" (Block Shuffle)
print("⚖️ Generating Safe Points...")
safe_df = fire_df.copy()
weather_cols = ['temp', 'humidity', 'wind']
# Block shuffle to keep physics real
safe_df[weather_cols] = fire_df[weather_cols].sample(frac=1).values
safe_df['fire_detected'] = 0

master_df = pd.concat([fire_df, safe_df], ignore_index=True)

# 4. SCIENTIFIC FEATURES (VPD & Time)
print("🧪 Calculating Scientific Features...")

# Vapor Pressure Deficit (VPD)
master_df['svp'] = 0.6108 * np.exp(17.27 * master_df['temp'] / (master_df['temp'] + 237.3))
master_df['vpd'] = master_df['svp'] * (1 - master_df['humidity'] / 100)

# Cyclical Month Encoding
master_df['acq_date'] = pd.to_datetime(master_df['acq_date'])
master_df['month'] = master_df['acq_date'].dt.month
master_df['month_sin'] = np.sin(2 * np.pi * master_df['month'] / 12)
master_df['month_cos'] = np.cos(2 * np.pi * master_df['month'] / 12)

# Save
os.makedirs('data/processed', exist_ok=True)
master_df = shuffle(master_df, random_state=42)
master_df.to_csv('data/processed/training_data_final.csv', index=False)
print(f"✅ Training Data Ready: {len(master_df)} rows")
print("👉 Saved to: data/processed/training_data_final.csv")

📂 Current Working Directory: c:\Users\vaast\Documents\Wildfire_Project
🔎 Looking for enriched files...
   ✅ Loaded: data/chunks\enriched_part_1_early.csv (33333 rows)
   ✅ Loaded: data/chunks\enriched_part_2_mid.csv (33323 rows)
   ✅ Loaded: data/chunks\enriched_part_3_late.csv (33328 rows)

🔥 Total verified fires: 99984
⚖️ Generating Safe Points...
🧪 Calculating Scientific Features...
✅ Training Data Ready: 199968 rows
👉 Saved to: data/processed/training_data_final.csv
